# DecodeLabs — Data Analytics Project 1
## Data Cleaning & Preparation
**Batch:** 2026 | **Powered by:** DecodeLabs

---
This notebook cleans the raw sales dataset by:
- Identifying and handling missing values (Phase 1: Strategic Imputation)
- Removing duplicate records (Phase 2: Integrity Audit)
- Standardising data formats — dates, numbers, and text (Phase 3: Speak One Language)


## 1. Setup — Import Libraries & Upload Dataset

In [1]:
from google.colab import files
uploaded = files.upload()

Saving Dataset for Data Analytics.xlsx to Dataset for Data Analytics.xlsx


In [2]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully.")

Libraries imported successfully.


## 2. Load Dataset

In [3]:
df = pd.read_excel('Dataset for Data Analytics.xlsx')
print(f"Dataset loaded successfully!")
print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns")

Dataset loaded successfully!
Shape: 1200 rows x 14 columns


## 3. Exploratory Data Analysis (EDA)
Quick overview of the dataset — structure, data types, and basic statistics.

In [4]:
# First 5 rows
df.head()

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [5]:
# Column names and data types
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [6]:
# Statistical summary of numeric columns
df.describe()

,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


## 4. Phase 1 — Strategic Imputation (Missing Values)
> *"Handle the gaps. Don't just delete."* — DecodeLabs

Identify which columns have missing values, then impute them with meaningful defaults
rather than dropping records (listwise deletion reduces statistical power).

In [7]:
# Check missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

Missing values per column:
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity             0
UnitPrice            0
ShippingAddress      0
PaymentMethod        0
OrderStatus          0
TrackingNumber       0
ItemsInCart          0
CouponCode         309
ReferralSource       0
TotalPrice           0
dtype: int64


In [8]:
# CouponCode has 309 missing values.
# These represent orders placed without a coupon — impute with 'No Coupon'.
df['CouponCode'] = df['CouponCode'].fillna('No Coupon')

# Verify — should now show 0
print(f"Missing values in CouponCode after imputation: {df['CouponCode'].isnull().sum()}")
print(f"Total missing values remaining: {df.isnull().sum().sum()}")

Missing values in CouponCode after imputation: 0
Total missing values remaining: 0


## 5. Phase 2 — Integrity Audit (Duplicate Records)
> *"One Truth, One Record."* — DecodeLabs

Every OrderID must be unique. Duplicate IDs inflate transaction counts and corrupt analysis.

In [9]:
# Check for duplicate OrderIDs
duplicate_count = df.duplicated(subset=['OrderID'], keep=False).sum()
print(f"Duplicate OrderID records found: {duplicate_count}")

# Remove duplicates if any exist — keep the first occurrence
df = df.drop_duplicates(subset=['OrderID'], keep='first')

# Verification Gate (required for Project 2 unlock)
remaining = df['OrderID'].duplicated().sum()
print(f"Duplicate OrderIDs after cleaning: {remaining}")
assert remaining == 0, "ERROR: Duplicate OrderIDs still exist!"
print("PASS — 0% error rate on Unique Identifiers.")

Duplicate OrderID records found: 0
Duplicate OrderIDs after cleaning: 0
PASS — 0% error rate on Unique Identifiers.


## 6. Phase 3 — Speak One Language (Format Standardisation)
> *"ISO 8601 Dates | Proper Case & Trim Whitespace | Numeric Precision (2 decimals)"* — DecodeLabs

All data must follow a single consistent format so downstream analysis is reliable.

### 6a. Date Format — ISO 8601 (datetime64)

In [10]:
# Convert Date column to datetime (ISO 8601 standard)
df['Date'] = pd.to_datetime(df['Date'])

# Verification Gate
assert 'datetime64' in str(df['Date'].dtype), "ERROR: Date column is not datetime!"
print(f"Date dtype: {df['Date'].dtype}")
print("PASS — 0% error rate on Date Formats.")

Date dtype: datetime64[ns]
PASS — 0% error rate on Date Formats.


### 6b. Numeric Precision — 2 Decimal Places

In [11]:
# Round both price columns to 2 decimal places
df['UnitPrice']  = df['UnitPrice'].round(2)
df['TotalPrice'] = df['TotalPrice'].round(2)

print("UnitPrice  — max decimal places:", df['UnitPrice'].apply(
    lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0).max())
print("TotalPrice — max decimal places:", df['TotalPrice'].apply(
    lambda x: len(str(x).split('.')[-1]) if '.' in str(x) else 0).max())
print("Both columns rounded to 2 decimal places.")

UnitPrice  — max decimal places: 2
TotalPrice — max decimal places: 2
Both columns rounded to 2 decimal places.


### 6c. Text Standardisation — Strip Whitespace & Proper Case

In [12]:
# Apply strip (remove leading/trailing spaces) and title case to text columns
text_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'ShippingAddress']

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

print("Text columns standardised:")
for col in text_cols:
    print(f"  {col}: {df[col].unique()[:4]} ...")

Text columns standardised:
  Product: ['Monitor' 'Phone' 'Tablet' 'Chair'] ...
  PaymentMethod: ['Debit Card' 'Online' 'Credit Card' 'Gift Card'] ...
  OrderStatus: ['Shipped' 'Cancelled' 'Returned' 'Delivered'] ...
  ReferralSource: ['Instagram' 'Referral' 'Email' 'Facebook'] ...
  ShippingAddress: ['928 Main St' '823 Main St' '512 Main St' '275 Main St'] ...


## 7. Final Dataset Verification
Full audit before export — confirming the dataset meets all DecodeLabs quality standards.

In [13]:
print("=" * 50)
print("FINAL DATASET AUDIT")
print("=" * 50)
print(f"Total rows             : {df.shape[0]}")
print(f"Total columns          : {df.shape[1]}")
print(f"Missing values         : {df.isnull().sum().sum()}")
print(f"Duplicate OrderIDs     : {df['OrderID'].duplicated().sum()}")
print(f"Date dtype             : {df['Date'].dtype}")
print(f"UnitPrice dtype        : {df['UnitPrice'].dtype}")
print(f"TotalPrice dtype       : {df['TotalPrice'].dtype}")
print("=" * 50)
print("Dataset is clean and ready for analysis.")

FINAL DATASET AUDIT
Total rows             : 1200
Total columns          : 14
Missing values         : 0
Duplicate OrderIDs     : 0
Date dtype             : datetime64[ns]
UnitPrice dtype        : float64
TotalPrice dtype       : float64
Dataset is clean and ready for analysis.


In [14]:
# Preview final clean dataset
df.head(10)

,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04
5,ORD200005,2023-10-23,C37249,Phone,2,245.86,934 Main St,Credit Card,Shipped,TRK72976927,4,SAVE10,Instagram,491.72
6,ORD200006,2025-06-17,C83492,Laptop,1,664.42,986 Main St,Gift Card,Returned,TRK96417362,6,SAVE10,Facebook,664.42
7,ORD200007,2023-05-12,C41460,Monitor,5,149.55,706 Main St,Cash,Shipped,TRK78809193,9,FREESHIP,Facebook,747.75
8,ORD200008,2025-04-02,C26817,Phone,2,134.28,904 Main St,Gift Card,Cancelled,TRK61042692,2,No Coupon,Email,268.56
9,ORD200009,2023-11-21,C31946,Desk,4,509.38,102 Main St,Credit Card,Shipped,TRK33478363,6,SAVE10,Google,2037.52


## 8. Save Cleaned Dataset
Exporting the cleaned dataset to CSV and Excel formats.
Files will be automatically downloaded to your local machine via Google Colab.


In [16]:

# ── Save as Excel ──────
df.to_excel('Cleaned_Data.xlsx', index=False)
print("Saved: Cleaned_Data.xlsx")

print(f"\nFile details:")
print(f"  Rows    : {df.shape[0]}")
print(f"  Columns : {df.shape[1]}")
print(f"  Missing : {df.isnull().sum().sum()}")


Saved: Cleaned_Data.xlsx

File details:
  Rows    : 1200
  Columns : 14
  Missing : 0


In [17]:
from google.colab import files

files.download('Cleaned_Data.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>